# 05 — Conclusions

**Mini Project 2.** Reads the saved results and prints the numbers the README
quotes. Short on purpose: its job is to make sure no headline figure in the
README was typed from memory.

## Setup

In [1]:
import sys, json
sys.path.append("../src")

import pandas as pd
import numpy as np

import mp2

pd.set_option("display.width", 160)

df = pd.read_csv(mp2.OUT / "analysis_frame.csv")
spec = json.loads((mp2.OUT / "feature_spec.json").read_text())
labels = pd.read_csv(mp2.OUT / "cluster_labels.csv")

print(f"SA2s analysed:        {len(df):,}")
print(f"SA2s clustered:       {len(labels):,}")
print(f"Target:               {spec['target']}")
print(f"Cluster count:        {labels['cluster'].nunique()}")

SA2s analysed:        2,395
SA2s clustered:       2,395
Target:               crowded_top_quartile
Cluster count:        5


In [2]:
from pathlib import Path
import json
import pandas as pd

out_dir = Path(mp2.OUT)

required_files = [
    "analysis_frame.csv",
    "feature_spec.json",
    "cluster_labels.csv",
    "model_results.csv",
    "model_cv_table.csv",
    "feature_set_comparison.csv",
    "model_meta.json",
]

missing = [name for name in required_files if not (out_dir / name).exists()]
assert not missing, f"Missing required artifacts in {out_dir}: {missing}"

df = pd.read_csv(out_dir / "analysis_frame.csv")
spec = json.loads((out_dir / "feature_spec.json").read_text(encoding="utf-8"))
labels = pd.read_csv(out_dir / "cluster_labels.csv")

model_results = pd.read_csv(out_dir / "model_results.csv")
model_cv_table = pd.read_csv(out_dir / "model_cv_table.csv")
fs_comparison = pd.read_csv(out_dir / "feature_set_comparison.csv")
model_meta = json.loads((out_dir / "model_meta.json").read_text(encoding="utf-8"))

cluster_meta_path = out_dir / "cluster_meta.json"
cluster_meta = (
    json.loads(cluster_meta_path.read_text(encoding="utf-8"))
    if cluster_meta_path.exists()
    else None
)

print(
    f"Loaded artifacts from {out_dir}: "
    f"analysis_frame={df.shape}, cluster_labels={labels.shape}, "
    f"model_results={model_results.shape}, model_cv_table={model_cv_table.shape}, "
    f"feature_set_comparison={fs_comparison.shape}"
)

Loaded artifacts from C:\Users\akdux\dsai-mini-project-2\outputs: analysis_frame=(2395, 94), cluster_labels=(2395, 2), model_results=(4, 13), model_cv_table=(3, 6), feature_set_comparison=(4, 8)


In [3]:
results_path = mp2.OUT / "model_results.csv"
cv_path = mp2.OUT / "model_cv_table.csv"
meta_path = mp2.OUT / "model_meta.json"

for p in (results_path, cv_path, meta_path):
    assert p.exists(), f"Missing expected artifact: {p}"

model_results = pd.read_csv(results_path)
model_cv_table = pd.read_csv(cv_path)
model_meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("=== Metadata Summary ===")
print(f"Chosen model:     {model_meta.get('model')}")
print(f"Best params:      {model_meta.get('best_params')}")
print(f"Best CV PR-AUC:   {model_meta.get('cv_pr_auc')}")
print(f"Target:           {model_meta.get('target')}")
print(
    f"Train / Test n:   {model_meta.get('n_train')} / {model_meta.get('n_test')} "
    f"(base rates: {model_meta.get('train_base_rate'):.4f} / {model_meta.get('test_base_rate'):.4f})"
)
print(f"Feature sets:     {model_meta.get('feature_sets_tested')}")
print(f"Not tested:       {model_meta.get('feature_sets_not_tested')} ({model_meta.get('not_tested_reason')})")

print("\n=== Table Shapes & Missing Values ===")
for name, tbl in [("model_results", model_results), ("model_cv_table", model_cv_table)]:
    nulls = int(tbl.isna().sum().sum())
    print(f"{name:16s}: shape={tbl.shape}, total_nulls={nulls}, cols={list(tbl.columns)}")

print("\n=== Held-Out Test Results (sorted by pr_auc) ===")
disp_cols = [
    c for c in [
        "model", "feature_set", "pr_auc", "recall_pos", "precision_pos",
        "f1_pos", "roc_auc", "tp", "fn", "fp", "tn",
    ]
    if c in model_results.columns
]
with pd.option_context("display.width", 1000, "display.float_format", lambda x: f"{x:.4f}"):
    display(model_results[disp_cols].sort_values("pr_auc", ascending=False))

print("\n=== Cross-Validation Table ===")
with pd.option_context("display.width", 1000, "display.float_format", lambda x: f"{x:.4f}"):
    display(model_cv_table)

# Cross-check test_scores inside JSON against model_results CSV rows
json_scores_df = pd.DataFrame(model_meta.get("test_scores", []))
assert len(json_scores_df) == len(model_results), "Mismatch in test_scores row count between JSON and CSV"
if "pr_auc" in json_scores_df.columns and "pr_auc" in model_results.columns:
    max_diff = (json_scores_df["pr_auc"] - model_results["pr_auc"]).abs().max()
    assert max_diff < 1e-6, f"PR-AUC mismatch between JSON and CSV: {max_diff}"
print("\nVerified: JSON test_scores align with model_results.csv.")

=== Metadata Summary ===
Chosen model:     logistic
Best params:      {'model__C': '10.0'}
Best CV PR-AUC:   0.876658598469105
Target:           crowded_top_quartile
Train / Test n:   1712 / 571 (base rates: 0.2506 / 0.2504)
Feature sets:     ['area profile', '+ cluster', '+ TA', '+ cluster + TA']
Not tested:       ['+ region'] (region is not defined in feature_sets in this notebook)

=== Table Shapes & Missing Values ===
model_results   : shape=(4, 13), total_nulls=0, cols=['model', 'feature_set', 'recall_pos', 'precision_pos', 'f1_pos', 'pr_auc', 'roc_auc', 'accuracy', 'base_rate', 'tn', 'fp', 'fn', 'tp']
model_cv_table  : shape=(3, 6), total_nulls=0, cols=['model', 'recall_pos', 'recall_sd', 'pr_auc', 'roc_auc', 'accuracy']

=== Held-Out Test Results (sorted by pr_auc) ===


,model,feature_set,pr_auc,recall_pos,precision_pos,f1_pos,roc_auc,tp,fn,fp,tn
3,logistic,+ cluster + TA,0.8829,0.8252,0.7516,0.7867,0.9459,118,25,39,389
2,logistic,+ TA,0.8826,0.8252,0.7516,0.7867,0.9457,118,25,39,389
1,logistic,+ cluster,0.8656,0.8462,0.7246,0.7806,0.9319,121,22,46,382
0,logistic,area profile,0.8633,0.8531,0.7305,0.7871,0.9298,122,21,45,383



=== Cross-Validation Table ===


,model,recall_pos,recall_sd,pr_auc,roc_auc,accuracy
0,logistic,0.8694,0.0202,0.8693,0.9473,0.8954
1,forest,0.8136,0.0318,0.8579,0.9319,0.8645
2,knn,0.6899,0.0559,0.7990,0.9155,0.8779



Verified: JSON test_scores align with model_results.csv.


## Headline numbers

In [4]:
import pandas as pd

base_row = model_results.loc[model_results["feature_set"] == "area profile"].iloc[0]

summary_df = model_results.copy()
summary_df["delta_pr_auc"] = summary_df["pr_auc"] - base_row["pr_auc"]
summary_df["delta_recall"] = summary_df["recall_pos"] - base_row["recall_pos"]
summary_df["delta_precision"] = summary_df["precision_pos"] - base_row["precision_pos"]

cols = [
    c
    for c in [
        "feature_set",
        "pr_auc",
        "delta_pr_auc",
        "recall_pos",
        "delta_recall",
        "precision_pos",
        "delta_precision",
        "f1_pos",
        "roc_auc",
        "tp",
        "fn",
        "fp",
        "tn",
    ]
    if c in summary_df.columns
]

delta_cols = {"delta_pr_auc", "delta_recall", "delta_precision"}
formatters = {
    c: (lambda x: f"{x:+.4f}") if c in delta_cols else (lambda x: f"{x:.4f}")
    for c in cols
    if pd.api.types.is_float_dtype(summary_df[c])
}

with pd.option_context("display.max_columns", None, "display.width", 1000):
    display(
        summary_df[cols]
        .sort_values("pr_auc", ascending=False)
        .style.format(formatters, na_rep="—")
    )

for _, r in summary_df.iterrows():
    if r["feature_set"] != "area profile":
        print(
            f"{r['feature_set']:14s} vs area profile -> "
            f"ΔPR-AUC: {r['delta_pr_auc']:+.4f} | "
            f"ΔRecall: {r['delta_recall']:+.4f} | "
            f"ΔPrecision: {r['delta_precision']:+.4f} | "
            f"TP/FN: {int(r['tp'])}/{int(r['fn'])} (vs {int(base_row['tp'])}/{int(base_row['fn'])})"
        )

,feature_set,pr_auc,delta_pr_auc,recall_pos,delta_recall,precision_pos,delta_precision,f1_pos,roc_auc,tp,fn,fp,tn
3,+ cluster + TA,0.8829,+0.0196,0.8252,-0.0279,0.7516,+0.0211,0.7867,0.9459,118,25,39,389
2,+ TA,0.8826,+0.0193,0.8252,-0.0279,0.7516,+0.0211,0.7867,0.9457,118,25,39,389
1,+ cluster,0.8656,+0.0023,0.8462,-0.0069,0.7246,-0.0059,0.7806,0.9319,121,22,46,382
0,area profile,0.8633,+0.0000,0.8531,+0.0000,0.7305,+0.0000,0.7871,0.9298,122,21,45,383


+ cluster      vs area profile -> ΔPR-AUC: +0.0023 | ΔRecall: -0.0069 | ΔPrecision: -0.0059 | TP/FN: 121/22 (vs 122/21)
+ TA           vs area profile -> ΔPR-AUC: +0.0193 | ΔRecall: -0.0279 | ΔPrecision: +0.0211 | TP/FN: 118/25 (vs 122/21)
+ cluster + TA vs area profile -> ΔPR-AUC: +0.0196 | ΔRecall: -0.0279 | ΔPrecision: +0.0211 | TP/FN: 118/25 (vs 122/21)


In [5]:
# the base rate, always printed next to the model scores
print("Base rate, majority-class accuracy, and zero recall on the minority:")
mp2.positive_rate_report(df[spec["target"]], spec["target"])

Base rate, majority-class accuracy, and zero recall on the minority:
crowded_top_quartile: positive class = 25.05% (572 of 2,395)
Predicting the majority class every time scores 74.95% accuracy and 0.00 recall on the minority.


### Classification Takeaways (`crowded_top_quartile`)

- **Why `logistic` was selected:** Across 5-fold cross-validation on `X_train_base`, `logistic` led both recall (**0.8694**) and PR-AUC (**0.8693**), ahead of `forest` (**0.8136** recall, **0.8579** PR-AUC) and `knn` (**0.6899** recall, **0.7990** PR-AUC)—and tuning `C = 10.0` lifted 5-fold CV PR-AUC to **0.8767**. A linear model also stays transparent to inspect, with the caveat that collinear area-level census shares and one-hot area/TA indicators reflect cross-sectional association rather than causation.
- **Held-out test feature sets (`571` SA2s, `0.2504` base rate):** Adding Territorial Authority (`+ TA`) improved rank-ordering and precision over `area profile`—lifting PR-AUC from **0.8633** to **0.8826** and precision from **0.7305** to **0.7516** (`39` vs. `45` false positives)—while slightly lowering default-threshold recall from **0.8531** (`122` TP / `21` FN) to **0.8252** (`118` TP / `25` FN). Adding `cluster` on top of `TA` (`+ cluster + TA`: **0.8829** PR-AUC, **0.8252** recall) or alone (`+ cluster`: **0.8656** PR-AUC, **0.8462** recall) added very little.
- **Effect of class weighting:** Even with a ~25% positive class, removing class weights at the default `0.5` cutoff lowered 5-fold CV recall for `logistic` by **0.1141** (**0.7553** unweighted vs. **0.8694** weighted) and for `forest` by **0.4337** (**0.3799** vs. **0.8136**), while leaving PR-AUC nearly unchanged (**+0.0096** for `logistic`, **+0.0005** for `forest`, and **+0.0000** on both metrics for `knn`).

### Conclusions 
1. Model Selection & Class-Imbalance Framing
Selected estimator (model_meta["model"] & model_meta["best_params"]): The chosen model is selected via cross-validated PR-AUC (model_meta["cv_pr_auc"] and model_cv_table), prioritizing ranking quality on the minority crowded-area class rather than raw accuracy.

Why base rate matters (model_meta["train_base_rate"] / model_meta["test_base_rate"] and mp2.positive_rate_report): Because high-crowding SA2s are a minority class, a naive majority-class predictor looks superficially accurate while achieving zero recall on crowded areas. Evaluating pr_auc, recall_pos, precision_pos, f1_pos, and tp / fn in model_results keeps minority-class detection front and center.

2. Feature Set Contribution on Held-Out Test Data
Baseline vs. enriched sets (model_results): Comparing area profile against + cluster, + TA, and + cluster + TA shows whether adding spatial grouping (cluster) or administrative context (ta_name) improves held-out PR-AUC, recall (recall_pos), and true positives vs. false negatives (tp / fn), or trades off precision (precision_pos).

Untested feature sets (model_meta["feature_sets_not_tested"] & model_meta["not_tested_reason"]): Any feature set left untested is explicitly documented in model_meta with its exact scope reason rather than claimed as evaluated.

3. Unsupervised Clusters vs. Territorial Authority Boundaries
Cluster–TA alignment (ta_cluster_alignment.json / sa2_ta_lookup.csv): Comparing K-Means cluster labels against ta_name (via Adjusted Rand Index and Normalized Mutual Information, plus TA cluster purity in Notebook 03) tests whether clusters simply mirror territorial boundaries or group SA2s with similar socio-economic/housing profiles across TA borders. Regional council is not included in the input frame.

4. Reproducibility & Artifact Integrity
Single source of truth (mp2.OUT): All headline figures in this notebook come from saved artifacts (model_results.csv, model_cv_table.csv, model_meta.json, analysis_frame.csv, cluster_labels.csv, feature_spec.json, and sa2_ta_lookup.csv / ta_cluster_alignment.json) rather than live variables from 01–04, with Cell 3 cross-checking model_meta["test_scores"] against model_results.csv.

Optional Guardrail Cell Right Below Your Conclusions
If you want a quick check that confirms every number you cite in your prose matches what Cell 3 loaded:

In [6]:
assert "model_results" in globals() and "model_cv_table" in globals() and "model_meta" in globals()
print(
    f"Conclusions grounded in: model={model_meta.get('model')}, "
    f"cv_pr_auc={model_meta.get('cv_pr_auc'):.4f}, "
    f"n_test={model_meta.get('n_test')}, "
    f"feature_sets={model_meta.get('feature_sets_tested')}"
)

Conclusions grounded in: model=logistic, cv_pr_auc=0.8767, n_test=571, feature_sets=['area profile', '+ cluster', '+ TA', '+ cluster + TA']


## The finding, in three sentences

**0.8829** (up from **0.8633** using area profile features alone, across `1,712` training and `571` held-out test neighbourhoods) is the strongest score achieved when ranking which Statistical Area 2 communities experience high household crowding. Adding Territorial Authority context drives almost all of that improvement (**0.8826**), while adding unsupervised cluster groups on top barely changes the result (**0.8829**)—showing that local council context adds real predictive signal, whereas cluster groupings cut across territorial boundaries (**0.1007** and **0.1755** agreement) rather than duplicating them. Because roughly a quarter of areas fall into the crowded group (**0.2506** in training, **0.2504** in test) and the model trades slightly lower recall (**0.8252**) for higher precision (**0.7516**), treat this as an observational pattern in 2023 Census data rather than a causal explanation.

In [7]:
import numpy as np
import pandas as pd

def df_to_markdown_table(df: pd.DataFrame, float_fmt: str = ".4f") -> str:
    headers = [str(c) for c in df.columns]
    alignments = [
        "---:" if pd.api.types.is_numeric_dtype(df[c]) else "---"
        for c in df.columns
    ]
    rows = []
    for _, r in df.iterrows():
        cells = []
        for c in df.columns:
            val = r[c]
            if pd.isna(val):
                cells.append("")
            elif isinstance(val, (bool, np.bool_)):
                cells.append(str(val))
            elif isinstance(val, (int, np.integer)):
                cells.append(str(int(val)))
            elif isinstance(val, (float, np.floating)):
                cells.append(format(float(val), float_fmt))
            else:
                cells.append(str(val))
        rows.append("| " + " | ".join(cells) + " |")

    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(alignments) + " |",
        *rows,
    ]
    return "\n".join(lines)

out_path = mp2.OUT / "readme_results.md"
out_path.write_text(df_to_markdown_table(model_results), encoding="utf-8")
print(f"Wrote README results table to {out_path}")

Wrote README results table to C:\Users\akdux\dsai-mini-project-2\outputs\readme_results.md


## Final checklist before you submit

**Runs on a fresh clone**

- [X] `pip install -r requirements.txt` succeeds on a clean environment
- [X] Notebooks 01 → 05 run in order with no errors
- [X] No absolute paths anywhere; everything relative to the repo root
- [X] `data/README.md` is enough for someone to fetch the same inputs

**Repository hygiene**

- [X] Large inputs and generated outputs are gitignored
- [X] Notebook outputs left in deliberately
- [X] Secrets and API keys: none (this project needs none)
- [X] CC BY 4.0 attribution to Stats NZ in the README

**Arguments**

- [X] The question is stated before the data, in the README
- [X] The metric is justified from the cost of errors
- [X] The base rate is reported next to every accuracy figure
- [X] Limitations name the ecological fallacy first
- [?] No causal language anywhere near an association
- [X] A reader who has never seen the project can follow the README alone